# HVAC Control with LLM + RL — Colab Pipeline

Runs the improved self-distillation pipeline (recommended best-of-N path):

```
ppo → select → bestofn → finetune (AWR) → eval (controlled)
```

`bestofn` uses the environment as verifier: the LLM proposes N actions per state
and the best (by true BEAR reward) is distilled. See `README.md` and
`METHODOLOGY_REVIEW.md`. Use an A100/T4 GPU runtime for the LLM stages
(`bestofn`, `finetune`, LLM evaluation).

## 1. Clone the repo and install dependencies

In [ ]:
REPO_URL = 'https://github.com/Mo119m/HAVC-control-with-reinforcement-learning-update.git'
BRANCH = 'claude/focused-cori-6TVCM'

import os
if not os.path.exists('HAVC-control-with-reinforcement-learning-update'):
    !git clone -b $BRANCH $REPO_URL
%cd HAVC-control-with-reinforcement-learning-update
!pip install -q -r requirements.txt

In [ ]:
# Verify the environment (deps, BEAR, GPU)
!python verify_environment.py

## 2. (Optional) Mount Google Drive for checkpoint backup

Pass `--drive_backup <path>` to the pipeline to resume across sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
DRIVE_BACKUP = ''  # e.g. '/content/drive/MyDrive/HVAC-RL-Backup'

## 3. Run the pipeline

Run all stages at once, or one by one (recommended so you can inspect each
output). `bestofn` proposes N actions per state and keeps the best by true BEAR
reward; `finetune` then trains on that data. Repeat `bestofn → finetune` for
expert iteration.

In [ ]:
DRIVE = ['--drive_backup', DRIVE_BACKUP] if DRIVE_BACKUP else []

# Option A: run everything
# !python core_modules/main_pipeline.py --stage all {' '.join(DRIVE)}

# Option B: run stage by stage (recommended best-of-N path)
!python core_modules/main_pipeline.py --stage ppo {' '.join(DRIVE)}
!python core_modules/main_pipeline.py --stage select {' '.join(DRIVE)}
!python core_modules/main_pipeline.py --stage bestofn {' '.join(DRIVE)}
!python core_modules/main_pipeline.py --stage finetune {' '.join(DRIVE)}

## 4. Controlled evaluation

Compare every controller on identical deterministic episodes. `zero` and `rule`
are cheap baselines; `ppo`, `llm`, `llm_ft` need the GPU.

In [ ]:
!python core_modules/evaluate.py \
    --controllers zero rule ppo llm llm_ft \
    --ppo_model pipeline_output/01_ppo_training/ppo_final.zip \
    --fewshot_json pipeline_output/02_few_shot_samples/few_shot_examples_structured.json \
    --episode_offsets 0 2000 4000 --max_steps 200 \
    --out_dir pipeline_output/06_eval

In [ ]:
# Show the comparison plot
from IPython.display import Image
Image('pipeline_output/06_eval/evaluation_comparison.png')

## 5. Generalization (the headline experiment)

One LLM, evaluated zero-shot across many buildings/climates, vs PPO (which cannot
even run on a building with a different zone count) and the rule baseline. This is
where the LLM has a structural advantage PPO cannot match.

In [ ]:
!python core_modules/generalization_eval.py \
    --controllers rule ppo llm llm_ft \
    --preset buildings \
    --ppo_model pipeline_output/01_ppo_training/ppo_final.zip \
    --adapter pipeline_output/04_finetuning/final_model \
    --fewshot_json pipeline_output/02_few_shot_samples/few_shot_examples_structured.json \
    --train_scenario OfficeSmall/Hot_Dry --max_steps 200 \
    --out_dir pipeline_output/07_generalization

from IPython.display import Image
Image('pipeline_output/07_generalization/generalization_comparison.png')